In [ ]:
from datasets import load_dataset

# Load and format data
dataset = load_dataset("csv", data_files='/content/drive/MyDrive/complex-simple-v1-anonymized.csv')

split_1 = dataset["train"].train_test_split(test_size=160, seed=42)
test_dataset = split_1["test"]
remaining_dataset = split_1["train"]


split_2 = remaining_dataset.train_test_split(test_size=150, seed=42)
eval_dataset = split_2["test"]
train_dataset = split_2["train"]

print(f"Train: {len(train_dataset)}")
print(f"Eval:  {len(eval_dataset)}")
print(f"Test:  {len(test_dataset)}")




In [ ]:
import json
import re

with open('jargon.json', 'r', encoding='utf-8') as f:
    jargon_data = json.load(f)

jargon_dict = {}
for entry in jargon_data:
    word = entry['word'].lower().strip()
    if entry['alternatives']:
        jargon_dict[word] = entry['alternatives'][0]

print(f"Loaded {len(jargon_dict)} jargon terms")
print(f"Example entries:")
for i, (k, v) in enumerate(list(jargon_dict.items())[:5]):
    print(f"  '{k}' -> '{v}'")

def lexical_simplify(text, jargon_dict):
    """
    Apply lexical simplification by replacing complex words/phrases
    with simpler alternatives from the jargon dictionary.

    Uses case-insensitive matching but preserves original casing pattern.
    """
    if not text or not isinstance(text, str):
        return text

    result = text

    # Sort by length to handle multi-word phrases before single words
    sorted_terms = sorted(jargon_dict.keys(), key=len, reverse=True)

    for term in sorted_terms:
        replacement = jargon_dict[term]

        # Create case-insensitive pattern with word boundaries
        # Use re.escape to handle special characters in terms
        pattern = r'\b' + re.escape(term) + r'\b'

        # Function to preserve case pattern
        def replace_with_case(match):
            matched_text = match.group(0)
            if matched_text.isupper():
                return replacement.upper()
            elif matched_text[0].isupper():
                return replacement.capitalize()
            else:
                return replacement.lower()

        result = re.sub(pattern, replace_with_case, result, flags=re.IGNORECASE)

    return result

In [ ]:
from tqdm import tqdm

# Prepare Data
sources = list(eval_dataset['Complex'])
references_flat = list(eval_dataset['Simple'])
references_sari = [[x] for x in references_flat]
references = references_flat

# Generate lexical simplification predictions
print("Generating Lexical Simplification Predictions...")
lexical_predictions = []
for text in tqdm(sources, desc="Lexical Simplification"):
    simplified = lexical_simplify(text, jargon_dict)
    lexical_predictions.append(simplified)


In [ ]:
import textstat
import numpy as np
from collections import Counter
from tqdm import tqdm
import re

def get_ngrams(text, n):
    if not text:
        return Counter()
    words = text.lower().split()
    if len(words) < n:
        return Counter()
    return Counter([tuple(words[i:i+n]) for i in range(len(words) - n + 1)])

def sari_score(source, prediction, references, n_max=4):
    def precision(cand, ref):
        if not cand: return 0.0
        return sum((cand & ref).values()) / sum(cand.values())

    def recall(cand, ref):
        if not ref: return 0.0
        return sum((cand & ref).values()) / sum(ref.values())

    def f1(p, r):
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    keep_scores, del_scores, add_scores = [], [], []

    for n in range(1, n_max + 1):
        src_ng = get_ngrams(source, n)
        pred_ng = get_ngrams(prediction, n)
        ref_ng = Counter()
        for ref in references:
            ref_ng |= get_ngrams(ref, n)

        # KEEP: in source AND in refs
        src_in_refs = src_ng & ref_ng
        pred_keep = pred_ng & src_ng
        keep_scores.append(f1(precision(pred_keep, src_in_refs), recall(pred_keep, src_in_refs)))

        # DELETE: in source but NOT in refs
        src_not_refs = src_ng - ref_ng
        pred_del = src_ng - pred_ng
        del_scores.append(f1(precision(pred_del, src_not_refs), recall(pred_del, src_not_refs)))

        # ADD: in refs but NOT in source
        refs_not_src = ref_ng - src_ng
        pred_add = pred_ng - src_ng
        add_scores.append(f1(precision(pred_add, refs_not_src), recall(pred_add, refs_not_src)))

    return (np.mean(keep_scores) + np.mean(del_scores) + np.mean(add_scores)) / 3.0 * 100

def compute_sari(sources, predictions, references_list):
    return np.mean([sari_score(s, p, r) for s, p, r in zip(sources, predictions, references_list)])

# Data
sources = list(eval_dataset['Complex'])
references_flat = list(eval_dataset['Simple'])
references_sari = [[x] for x in references_flat]

# Sari
print("Calculating SARI...")
sari_src = compute_sari(sources, sources, references_sari)
sari_ref = compute_sari(sources, references_flat, references_sari)
sari_lex = compute_sari(sources, lexical_predictions, references_sari)

# Flesch Douma
print("Calculating Flesch-Douma...")
textstat.set_lang("nl_NL")
fd = lambda t: np.mean([textstat.flesch_reading_ease(x) for x in t if x and len(x.strip()) > 5])
fd_src, fd_ref, fd_lex = fd(sources), fd(references_flat), fd(lexical_predictions)

# Lint 2
print("Calculating LiNT-II...")
try:
    from lint_ii import ReadabilityAnalysis
    import logging
    logging.getLogger("lint_ii").setLevel(logging.ERROR)

    def lint(texts):
        scores, levels = [], []
        for t in tqdm(texts):
            try:
                a = ReadabilityAnalysis.from_text(t)
                if a.lint.score is not None:
                    scores.append(a.lint.score)
                    levels.append(a.lint.level)
            except: pass
        return (np.mean(scores), np.mean(levels)) if scores else (0, 0)

    lint_s_src, lint_l_src = lint(sources)
    lint_s_ref, lint_l_ref = lint(references_flat)
    lint_s_lex, lint_l_lex = lint(lexical_predictions)
    LINT_OK = True
except Exception as e:
    print(f"LiNT-II failed: {e}")
    lint_s_src = lint_l_src = lint_s_ref = lint_l_ref = lint_s_lex = lint_l_lex = float('nan')
    LINT_OK = False

# Avg len
sl = lambda t: np.mean([len(re.findall(r'\w+', x)) for x in t if x])
sl_src, sl_ref, sl_lex = sl(sources), sl(references_flat), sl(lexical_predictions)

# Results
print("Lexical Simplification Results")
print(f"{'METRIC':<20} | {'SOURCE':<11} | {'REFERENCE':<11} | {'LEXICAL':<11}")
print(f"{'SARI ↑':<20} | {sari_src:<11.2f} | {sari_ref:<11.2f} | {sari_lex:<11.2f}")
print(f"{'Flesch-Douma ↑':<20} | {fd_src:<11.2f} | {fd_ref:<11.2f} | {fd_lex:<11.2f}")
if LINT_OK:
    print(f"{'LiNT-II Score ↓':<20} | {lint_s_src:<11.2f} | {lint_s_ref:<11.2f} | {lint_s_lex:<11.2f}")
    print(f"{'LiNT-II Level ↓':<20} | {lint_l_src:<11.2f} | {lint_l_ref:<11.2f} | {lint_l_lex:<11.2f}")
print(f"{'Avg Words/Sent':<20} | {sl_src:<11.2f} | {sl_ref:<11.2f} | {sl_lex:<11.2f}")

print(f"\nJargon terms: {len(jargon_dict)}")
print(f"Sentences modified: {changed_count}/{len(sources)} ({100*changed_count/len(sources):.1f}%)")

# Deltas
print(f"{'Flesch-Douma':<20} | Lex: {fd_lex-fd_src:>+.2f} | Ref: {fd_ref-fd_src:>+.2f}")
if LINT_OK:
    print(f"{'LiNT-II Level':<20} | Lex: {lint_l_lex-lint_l_src:>+.2f} | Ref: {lint_l_ref-lint_l_src:>+.2f}")
print(f"{'Words/Sent':<20} | Lex: {sl_lex-sl_src:>+.2f} | Ref: {sl_ref-sl_src:>+.2f}")